<a href="https://colab.research.google.com/github/xuebingwu/kpLogo/blob/master/kplogo_colab_form.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# *k*pLogo: *k*-mer probability logo

This notebook replicates the [kpLogo web server form](https://kplogo.wi.mit.edu/submit.html) so you can run kpLogo interactively in Google Colab. The same kpLogo program (version 1.0) and the same command line are used, so results are identical to the web server.

**How to use:**
1. **Runtime > Run all** (or run each cell in order). The first run installs the plotting dependencies and downloads the kpLogo binary (~1 min).
2. Fill in the job form in the **"Job Form"** cell (click the small form icon on the left of the cell if the fields are hidden).
3. Run the **"Run kpLogo"** cell. Logo figures are displayed below it and all output files are bundled into a downloadable zip.

**Notes**
- For more than a couple of short sequences, choose `upload a file` as the foreground source (FASTA, raw, or tabular format — see the [manual](https://kplogo.wi.mit.edu/manual.html)).
- The web server's optional email notification is not available here; results are downloaded directly instead.
- The web form's *predefined Markov model* background (hg/ht/hp/mg/mt/mp) is not implemented by the web server either, so it is omitted here.
- Output files are kept in the Colab workspace (deleted when the runtime resets) — download anything you want to keep.

## Job Form

Defaults match the web server. Hover any field to see its meaning; full documentation is in the [manual](https://kplogo.wi.mit.edu/manual.html).

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os, subprocess, sys

# plotting dependencies used by kpLogo (ps2pdf: ghostscript, convert: imagemagick)
subprocess.run("apt-get install -y -qq ghostscript imagemagick > /dev/null 2>&1", shell=True)

if not os.path.exists("kpLogo"):
    print("Downloading kpLogo binary ...")
    r = subprocess.run("curl -sL -o kpLogo "
        "https://raw.githubusercontent.com/xuebingwu/kpLogo/master/bin/kpLogo", shell=True)
    if r.returncode != 0 or not os.path.exists("kpLogo") or os.path.getsize("kpLogo") < 100000:
        print("Direct download failed; falling back to cloning the repository ...")
        subprocess.run("git clone --depth 1 https://github.com/xuebingwu/kpLogo.git", shell=True, check=True)
        os.replace("kpLogo/bin/kpLogo", "kpLogo")
        subprocess.run("rm -rf kpLogo_repo", shell=True)
    os.chmod("kpLogo", 0o755)

KPLOGO_BIN = os.path.abspath("kpLogo")

subprocess.run("./kpLogo 2>&1 | head -3", shell=True)
print("Setup complete. kpLogo binary: " + KPLOGO_BIN)

In [ ]:
#@title Job Form { display-mode: "form" }

#@markdown ### Input
foreground_source = "paste sequences" #@param ["paste sequences", "upload a file"]
foreground_sequences = "" #@param {type:"string"}
#@markdown *Sequences to paste (if `paste sequences` is selected). Separate sequences with actual line breaks or with the two characters `\n`. For real datasets use `upload a file` (FASTA / raw / tabular).*

inputtype = "(default) binomial test" #@param ["(default) binomial test", "-ranked", "-weighted"]
col_seq = 1 #@param {type:"integer"}
col_weight = 2 #@param {type:"integer"}
#@markdown *For tabular input: column containing the sequence and the weight/rank.*

region_first = 1 #@param {type:"integer"}
region_last = 0 #@param {type:"integer"}
#@markdown *Only consider positions `region_first`..`region_last` (0 or negative = distance from the end).*

select_pkmers = "" #@param {type:"string"}
remove_pkmers = "" #@param {type:"string"}
#@markdown *Comma-separated k-mers to keep or remove, e.g. `CNNC:47:0,UGCA:30:1`.*

#@markdown ### Alphabet and k-mers
alphabet = "dna" #@param ["dna", "protein", "other"]
other_alphabet = "" #@param {type:"string"}
#@markdown *If `other`: enter all allowed residues, e.g. `ACGTN` or `ACDEFGHIKLMNPQRSTVWY`.*

kmer_length = "-max_k 4" #@param ["-max_k 6", "-max_k 5", "-max_k 4", "-max_k 3", "-max_k 2", "-k 6", "-k 5", "-k 4", "-k 3", "-k 2", "-k 1"]
degenerate = "-gapped" #@param ["-gapped", "-degenerate all", "other", "(none)"]
degenerate_alphabet = "" #@param {type:"string"}
#@markdown *If `other`: enter the allowed degenerate symbols (subset of `ACGTRYMKWSBDHVN`). Ignored unless alphabet is `dna`.*

shift = "-shift 0" #@param ["-shift 0", "-shift 1", "-shift 2", "-shift 3", "-shift 4", "-shift 5", "-shift 6", "-max_shift 1", "-max_shift 2", "-max_shift 3", "-max_shift 4", "-max_shift 5", "-max_shift 6"]

#@markdown ### Background model (ignored for -ranked / -weighted)
background = "(default) same kmer at other positions" #@param ["(default) same kmer at other positions", "bgfile: provide background sequences", "shuffle: shuffle input sequences", "markov_foreground: markov model trained on foreground", "markov_background: markov model trained on background file", "markov_string: specify markov model as text"]
background_source = "paste sequences" #@param ["paste sequences", "upload a file"]
background_sequences = "" #@param {type:"string"}
#@markdown *Background sequences (used by `bgfile` and `markov_background`).*

shuffle_n = 100 #@param {type:"integer"}
shuffle_m = "2" #@param ["1", "2", "3", "4"] {type:"raw"}
markov_foreground_order = "1" #@param ["0", "1", "2"] {type:"raw"}
markov_background_order = "1" #@param ["0", "1", "2"] {type:"raw"}
markov_string = "A:0.25,C:0.25,G:0.25,T:0.25" #@param {type:"string"}

#@markdown ### Statistics and output
mincount = 0.01 #@param {type:"number"}
#@markdown *Minimum count of a k-mer to be included (<1 = fraction of sequences).*

pseudo = 1 #@param {type:"number"}
p = 0.01 #@param {type:"number"}
#@markdown *p-value cut-off (Bonferroni-corrected).*

maxFrac = 0.75 #@param {type:"number"}
#@markdown *Fix a position with a specific residue if it occurs in more than this fraction of sequences.*

small_sample = False #@param {type:"boolean"}
colorblind = False #@param {type:"boolean"}
last_residue = False #@param {type:"boolean"}
startPos = 1 #@param {type:"integer"}
plottype = "-plot p" #@param ["-plot p", "-plot b", "-plot s"]
stack_order = "-stack_order 1" #@param ["-stack_order 1", "-stack_order -1", "-stack_order 0"]
jobname = "my kpLogo job" #@param {type:"string"}

In [ ]:
#@title Run kpLogo { display-mode: "form" }
import os, re, subprocess, glob, zipfile

# locate the kpLogo binary (set by the Setup cell)
KPLOGO_BIN = globals().get("KPLOGO_BIN")
if not KPLOGO_BIN or not os.path.exists(KPLOGO_BIN):
    for cand in ["/content/kpLogo", "../kpLogo", "./kpLogo"]:
        if os.path.exists(cand):
            KPLOGO_BIN = cand
            break
    else:
        raise SystemExit("kpLogo binary not found - please run the Setup cell first.")

# same sanitization as the web server (web/submit.php)
def scrub_alphanum(s):
    if s is None: s = ""
    s = re.sub(r"\.+", ".", s)                       # all repeating dots to single dots
    s = re.sub(r"[^@0-9a-zA-Z\-_:,. ]", "", s)        # keep alphanum, comma, period, at, hyphen
    return s

def die(msg):
    print("\033[91m" + msg + "\033[0m")
    raise SystemExit(1)

jobname = scrub_alphanum(jobname) or "kpLogo job"
jobdir = "kpLogo_job_" + re.sub(r"[^0-9a-zA-Z\-_]", "_", jobname)
os.makedirs(jobdir, exist_ok=True)
os.chdir(jobdir)

# ---------- foreground input ----------
if foreground_source == "paste sequences":
    seqs = foreground_sequences.replace("\\n", "\n")
    if len(seqs.strip()) == 0:
        die("Please paste your sequences or choose 'upload a file'.")
    with open("kpLogo.input.txt", "w") as f:
        f.write(seqs)
else:
    print("Please upload your foreground sequence file (FASTA / raw / tabular):")
    try:
        from google.colab import files
        up = files.upload()
    except ImportError:  # local Jupyter fallback
        up = None
        fn = input("path to file: ").strip()
        up = {os.path.basename(fn): open(fn, "rb").read()}
    if not up:
        die("No file uploaded.")
    with open("kpLogo.input.txt", "wb") as f:
        f.write(list(up.values())[0])

# ---------- background input ----------
bg = ""
if background == "bgfile: provide background sequences":
    bg = " -bgfile kpLogo.background.txt "
elif background == "markov_foreground: markov model trained on foreground":
    bg = " -markov %s " % markov_foreground_order
elif background == "shuffle: shuffle input sequences":
    bg = " -shuffle %s,%s " % (shuffle_n, shuffle_m)
elif background == "markov_background: markov model trained on background file":
    bg = " -markov %s -bgfile kpLogo.background.txt " % markov_background_order
elif background == "markov_string: specify markov model as text":
    bg = " -markov %s" % scrub_alphanum(markov_string)

if background in ("bgfile: provide background sequences",
                  "markov_background: markov model trained on background file"):
    if background_source == "paste sequences":
        bseqs = background_sequences.replace("\\n", "\n")
        if len(bseqs.strip()) == 0:
            die("Please paste your background sequences or choose 'upload a file'.")
        with open("kpLogo.background.txt", "w") as f:
            f.write(bseqs)
    else:
        print("Please upload your background sequence file:")
        try:
            from google.colab import files
            up = files.upload()
        except ImportError:
            fn = input("path to file: ").strip()
            up = {os.path.basename(fn): open(fn, "rb").read()}
        if not up:
            die("No file uploaded.")
        with open("kpLogo.background.txt", "wb") as f:
            f.write(list(up.values())[0])

# ---------- options (same mapping as web/submit.php) ----------
it = {"(default) binomial test": "", "-ranked": "-ranked", "-weighted": "-weighted"}[inputtype]

alphabet_v = scrub_alphanum(alphabet)
if alphabet_v == "other":
    alphabet_v = scrub_alphanum(other_alphabet)
    if alphabet_v == "":
        die("Please enter the allowed residues for the 'other' alphabet.")

deg = scrub_alphanum(degenerate)
if deg == "other":
    da = scrub_alphanum(degenerate_alphabet)
    if da == "":
        die("Please enter degenerate symbols!")
    deg = " -degenerate %s " % da
elif deg == "(none)":
    deg = ""
if alphabet_v != "dna":
    deg = ""  # same rule as the web server

sel = (" -select " + scrub_alphanum(select_pkmers)) if scrub_alphanum(select_pkmers) != "" else ""
rem = (" -remove " + scrub_alphanum(remove_pkmers)) if scrub_alphanum(remove_pkmers) != "" else ""

mincount_v = mincount if mincount >= 0 else 0  # same clamp as the web server

# ---------- command (identical construction to web/submit.php) ----------
cmd = (KPLOGO_BIN + " kpLogo.input.txt -o kpLogo.output %s -seq %s -weight %s -alphabet %s  %s %s %s "
       "-startPos %s %s %s -minCount %s -pseudo %s -region %s,%s %s %s %s %s -pc %s %s %s -fix %s"
       ) % (it, scrub_alphanum(str(col_seq)), scrub_alphanum(str(col_weight)), alphabet_v,
            kmer_length, shift, bg, scrub_alphanum(str(startPos)), deg,
            "-colorblind" if colorblind else "",
            scrub_alphanum(str(mincount_v)), scrub_alphanum(str(pseudo)),
            scrub_alphanum(str(region_first)), scrub_alphanum(str(region_last)),
            sel, rem, plottype,
            "-small_sample" if small_sample else "",
            scrub_alphanum(str(p)),
            "-last_residue" if last_residue else "",
            stack_order, scrub_alphanum(str(maxFrac)))

print("Running:", cmd.replace(KPLOGO_BIN, "kpLogo"))
r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
with open("log", "w") as f:
    f.write(r.stdout + r.stderr)

if os.path.exists("exit_with_error"):
    print("\033[91mExit due to error!\033[0m")
    print(r.stdout + r.stderr)
    raise SystemExit(1)

# ---------- display results ----------
try:
    from IPython.display import Image, display
    shown = False
    for title, suffix in [("Frequency Logo", "freq"),
                          ("Information Content Logo", "info"),
                          ("k-mer Logo (most significant k-mer at each position)", "most.significant.each.position"),
                          ("Probability Logo", "")]:
        png = "kpLogo.output%s.png" % (("." + suffix) if suffix else "")
        if os.path.exists(png):
            shown = True
            print("\n" + title)
            display(Image(png, width=750))
    if not shown:
        print("No logo figures were produced (no significant k-mers found). See log below.")
except ImportError:
    pass

print("\nTabular output:")
for txt in ["kpLogo.output.most.significant.each.position.txt", "kpLogo.output.pass.p.cutoff.txt"]:
    if os.path.exists(txt):
        print("  " + txt)
print("\nScreen output:")
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)

# ---------- bundle for download ----------
out = jobname + ".zip"
with zipfile.ZipFile(out, "w") as z:
    for f in sorted(glob.glob("kpLogo.output.*")) + ["kpLogo.input.txt", "log"]:
        if os.path.exists(f):
            z.write(f)
print("\nAll output files were saved to " + out + " (in " + os.getcwd() + ")")
try:
    from google.colab import files
    files.download(out)
except ImportError:
    pass